# モデルの初期化

In [1]:
from langchain.chat_models import init_chat_model
import os
from pydantic import BaseModel, Field
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.messages import HumanMessage
from rich import print
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.tools import tool
from typing import Dict, Any
# from rich import print

load_dotenv(override=True)

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_API_BASE = os.getenv("OPENROUTER_API_BASE")

model = init_chat_model(
    model="openai/gpt-5.4-mini",
    model_provider="openai",
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_API_BASE,
)

## 関連ツールの定義
agentの機能を体現する


In [2]:
# ==================== ツール定義 ====================
from langchain_core.tools import tool
import math
from datetime import datetime, timedelta


@tool
def get_weather(city: str) -> str:
    """指定された都市のリアルタイム天気情報を取得する。中国の主要都市の天気照会に対応

    Args:
        city: 都市名、例えば"北京"、"上海"、"深圳"など

    Returns:
        気温、天気状況、大気質を含む詳細情報

    Examples:
        get_weather("北京") は "曇り、15-22℃、大気質良好" を返す
    """
    weather_db = {
        "北京": "曇り、15-22℃、大気質良好、湿度45%",
        "上海": "晴れ、18-25℃、大気質優良、湿度60%",
        "深圳": "小雨、22-28℃、大気質優良、湿度75%",
        "成都": "曇天、16-23℃、大気質良好、湿度70%",
        "杭州": "晴れ、17-24℃、大気質優良、湿度55%",
        "广州": "曇り、21-29℃、大気質良好、湿度72%"
    }
    result = weather_db.get(city)
    if result:
        return f"{city}：{result}"
    else:
        return f"申し訳ございません、現在{city}の天気情報の照会には対応していません。現在対応しているのは：北京、上海、深圳、成都、杭州、广州"


@tool
def calculator(expression: str) -> str:
    """数学計算を実行する。基本演算子（+、-、*、/、**）と一般的な数学関数に対応

    Args:
        expression: 数式。以下を含むことができる：
        - 基本演算：2 + 3, 10 * 5, 100 / 4
        - べき乗演算：2 ** 10
        - 関数：sqrt(16), abs(-5), pow(2, 3)

    Returns:
        計算結果またはエラー情報

    Examples:
        calculator("2 + 3 * 4") は "14" を返す
        calculator("sqrt(16)") は "4.0" を返す
    """
    try:
        # 安全な数学演算環境
        safe_functions = {
            "sqrt": math.sqrt,
            "pow": pow,
            "abs": abs,
            "round": round,
            "sin": math.sin,
            "cos": math.cos,
            "tan": math.tan,
            "log": math.log,
            "pi": math.pi,
            "e": math.e
        }
        result = eval(expression, {"__builtins__": {}}, safe_functions)
        return f"{expression} = {result}"
    except Exception as e:
        return f"計算エラー：{str(e)}\nヒント：数式の形式を確認してください。対応関数は sqrt,abs, pow, sin, cos, tan, log"


@tool
def get_time_info(query_type: str = "current") -> str:
    """時間に関する情報を取得する

    Args:
        query_type: 照会タイプ
        - "current": 現在時刻
        - "date": 今日の日付
        - "tomorrow": 明日の日付
        - "yesterday": 昨日の日付
        - "weekday": 曜日

    Returns:
        時間情報の文字列

    Examples:
        get_time_info("current") は "2025年1月25日 14:30:25" を返す
        get_time_info("weekday") は "土曜日" を返す
    """
    now = datetime.now()
    if query_type == "current":
        return now.strftime("現在時刻：%Y年%m月%d日 %H:%M:%S")
    elif query_type == "date":
        return now.strftime("今日は：%Y年%m月%d日")
    elif query_type == "tomorrow":
        tomorrow = now + timedelta(days=1)
        return tomorrow.strftime("明日は：%Y年%m月%d日")
    elif query_type == "yesterday":
        yesterday = now - timedelta(days=1)
        return yesterday.strftime("昨日は：%Y年%m月%d日")
    elif query_type == "weekday":
        weekdays = ["月曜日", "火曜日", "水曜日", "木曜日", "金曜日", "土曜日", "日曜日"]
        return f"今日は{weekdays[now.weekday()]}です"
    else:
        return f"対応していない照会タイプです：{query_type}。対応：current, date, tomorrow,yesterday, weekday"


@tool
def convert_currency(amount: float, from_curr: str, to_curr: str) -> str:
    """通貨換算ツール
    主要通貨間のリアルタイム為替レート換算に対応

    Args:
        amount: 金額
        from_curr: 換算元の通貨コード（CNY/USD/EUR/GBP/JPY/HKD）
        to_curr: 換算先の通貨コード（CNY/USD/EUR/GBP/JPY/HKD）

    Returns:
        換算結果

    Examples:
    convert_currency(100, "CNY", "USD") は "100 CNY = 14.00 USD" を返す
    """
    # 為替レート表（CNY基準）
    exchange_rates = {
        "CNY": 1.0,  # 人民元
        "USD": 0.14,  # 米ドル
        "EUR": 0.13,  # ユーロ
        "GBP": 0.11,  # 英ポンド
        "JPY": 20.8,  # 日本円
        "HKD": 1.09  # 香港ドル
    }
    # 通貨名
    currency_names = {
        "CNY": "人民元", "USD": "米ドル", "EUR": "ユーロ",
        "GBP": "英ポンド", "JPY": "日本円", "HKD": "香港ドル"
    }
    from_curr = from_curr.upper()
    to_curr = to_curr.upper()
    if from_curr not in exchange_rates:
        return f"対応していない換算元通貨です：{from_curr}。対応通貨：CNY, USD, EUR, GBP,    JPY, HKD"
    if to_curr not in exchange_rates:
        return f"対応していない換算先通貨です：{to_curr}。対応通貨：CNY, USD, EUR, GBP,    JPY, HKD"
    # 換算ロジック：まずCNYに換算し、次に目標通貨に換算する
    cny_amount = amount / exchange_rates[from_curr]
    result_amount = cny_amount * exchange_rates[to_curr]
    from_name = currency_names[from_curr]
    to_name = currency_names[to_curr]
    return f"{amount} {from_name}（{from_curr}）= {result_amount:.2f}{to_name}（{to_curr}）"


@tool
def search_info(keyword: str, category: str = "all") -> str:
    """各種情報を検索する

    Args:
        keyword: 検索キーワード
        category: 検索カテゴリ
        - "product": 商品を検索
        - "news": ニュースを検索
        - "all": すべてを検索

    Returns:
        検索結果
    """
    # データベースをシミュレート
    products = {
        "スマホ": "iPhone 15 (¥5999), シャオミ14 (¥3999), ファーウェイMate60 (¥6999)",
        "ノートパソコン": "MacBook Pro (¥12999), ThinkPad X1 (¥9999), ファーウェイMateBook(¥7999)",
        "イヤホン": "AirPods Pro (¥1999), Sony WH-1000XM5 (¥2499)"
    }
    news = {
        "AI": "1. GPT-5がまもなくリリース 2. AIチップ市場が30%成長 3. 新しいAI規制が施行",
        "テクノロジー": "1. 量子コンピューティングの新たな突破 2. 6G技術のテスト 3. 新エネルギー車の販売台数が過去最高"
    }
    results = []
    if category in ["product", "all"]:
        for key, value in products.items():
            if keyword in key:
                results.append(f"【商品】{key}：{value}")
    if category in ["news", "all"]:
        for key, value in news.items():
            if keyword in key or keyword in value:
                results.append(f"【ニュース】{key} 関連：{value}")
    if results:
        return "\n".join(results)
    else:
        return f"'{keyword}' に関する{category}情報が見つかりませんでした"


## agentの作成、またはインスタンス化
model, tools,system_prompt,ToolStrategy(構造化出力を体現）;invoke--->メッセージリスト


In [3]:
from langchain.agents import create_agent


class SmartAssistant:
    """多機能インテリジェントアシスタント"""

    def __init__(self):
        # モデルを初期化
        self.model = model
        # ツールリスト
        self.tools = [
            get_weather,
            calculator,
            get_time_info,
            convert_currency,
            search_info
        ]
        # システムプロンプト
        system_prompt = """あなたは多機能なインテリジェントアシスタントで、ユーザーを次のように支援できます：
        🌤 天気照会：get_weather ツールを使用
        🔢 数学計算：calculator ツールを使用
        ⏰ 時刻照会：get_time_info ツールを使用
        💱 通貨換算：convert_currency ツールを使用
        🔍 情報検索：search_info ツールを使用

        重要な注意事項：
        1. ユーザーの質問をよく読み、どのツールを使う必要があるか判断する
        2. 複数のツールが必要な場合は、順番に呼び出す
        3. 常に親しみやすく、プロフェッショナルな口調で回答する
        4. ツールがデータを返した場合は、わかりやすい言葉でユーザーに説明する
        5. タスクを完了できない場合は、正直にその理由をユーザーに伝える
        常に日本語で回答してください。"""

        # ✅ agent を作成
        self.agent = create_agent(
            model=self.model,
            tools=self.tools,
            system_prompt=system_prompt
        )
        # 会話履歴
        self.messages = []

    def chat(self, user_input: str) -> str:
        """対話インターフェース"""
        # ユーザーメッセージを追加
        self.messages.append({"role": "user", "content": user_input})
        # agent を呼び出す
        result = self.agent.invoke({"messages": self.messages})
        # メッセージ履歴を更新
        self.messages = result["messages"]
        # 最後の AI メッセージを返す
        for msg in reversed(self.messages):
            if msg.type == "ai" and msg.content:
                return msg.content
        return "申し訳ございません、このリクエストを処理できませんでした。"

    def reset(self):
        """会話履歴をリセットする"""
        self.messages = []


#  メインプログラム、テストを実行

In [4]:
import sys
# ==================== メインプログラム ====================
def main():
    assistant = SmartAssistant()
    print("=" * 40)
    print("🤖 多機能インテリジェントアシスタント（LangChain 1.2）")
    print("=" * 40)
    print("\n私ができること：")
    print(" 🌤 天気照会")
    print(" 🔢 数学計算")
    print(" ⏰ 時刻照会")
    print(" 💱 通貨換算")
    print(" 🔍 情報検索")
    print("\n'quit' と入力して終了、'reset' と入力して会話をリセット\n")

    demos = [
        "北京の今日の天気はどうですか？",
        "(25 + 17) * 3 を計算してください",
        "今何時ですか？",
        "100米ドルは人民元でいくらですか？"
    ]

    for demo in demos:
        print(f"👤 {demo}")
        response = assistant.chat(demo)
        print(f"🤖 {response}\n")
    # 会話をリセット
    assistant.reset()

    # 対話モード
    print("=" * 40)
    print("💬 対話モードに入ります")
    print("=" * 40)


    while True:
        sys.stdout.flush()
        user_input = input("\n👤 あなた: ")
        if user_input.lower() == 'quit':
            print("さようなら！👋")
            break
        if user_input.lower() == 'reset':
            assistant.reset()
            print("✅ 会話をリセットしました")
            continue
        if not user_input.strip():
            continue
        # アシスタントを呼び出す
        response = assistant.chat(user_input)
        print(f"🤖 アシスタント: {response}")
        # sys.stdout.flush()

if __name__ == '__main__':
    main()


========================================

🤖 多機能インテリジェントアシスタント（LangChain 1.2）

========================================

私ができること：

🌤 天気照会

...

🤖 アシスタント: こんにちは！どうされましたか？

🤖 アシスタント: この会話で最初の質問は、**「1+1=」** です。

🤖 アシスタント: この会話の最後の質問は、**「最後の質問は？」** です。

🤖 アシスタント: 北京の天気は**曇り**で、**15〜22℃**、**大気質は良好**、**湿度は45%**です。

さようなら！👋